# Spleen segmentation (MSD Task09) with a 3D U-Net on Colab

**For education only, not for clinical use.**

MONAI `UNet` (spatial_dims=3), fixed config `configs/segmentation.yaml`. Preprocessing: RAS,
1.5 mm spacing, CT soft-tissue window. Training: random 96^3 crops and light augmentation (train
only), Dice + cross-entropy. Validation: sliding-window Dice on whole volumes; the best checkpoint
has the highest val mean Dice.

- **Not run yet.** The code is tested on tiny synthetic volumes on CPU only.
- Use a **GPU runtime** (T4 is enough). The cached volumes need about 3 GB of RAM.
- **Storage:** the dataset (about 1.5 GB, md5 checked) is downloaded to the VM disk, and training
  reads only from there. Drive holds the checkpoints (about 58 MB every 10 epochs, about 1.2 GB
  in all) and the report. Nothing on Drive is deleted.
- **Resumable:** after a disconnect, run all cells again; training continues after the latest
  checkpoint (at most 10 epochs are lost).

## 1. GPU, Drive and disk

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def run(*args, cwd=None, env=None):
    """Run a command, stream its output into the notebook, fail on a non-zero exit."""
    process = subprocess.Popen(
        [str(arg) for arg in args], cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True,
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"{args[0]} failed with exit code {process.returncode}")


def free_gb(path):
    return shutil.disk_usage(path).free / 1e9


import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

from google.colab import drive

drive.mount("/content/drive")
run("df", "-h", "/content", "/content/drive")
NEED_VM_GB = 6  # tar 1.5 GB + extracted 1.6 GB, with margin
NEED_DRIVE_GB = 1.5  # 20 checkpoints of about 58 MB, with margin
print(f"VM: {free_gb('/content'):.1f} GB free, {NEED_VM_GB} GB needed")
print(f"Drive: {free_gb('/content/drive'):.1f} GB free, {NEED_DRIVE_GB} GB needed")
if free_gb("/content") < NEED_VM_GB:
    raise RuntimeError("Not enough VM disk. Stop here.")
if free_gb("/content/drive") < NEED_DRIVE_GB:
    raise RuntimeError("Not enough Drive space for the checkpoints. Stop here.")

## 2. Code from the repo, DVC and MONAI

In [ ]:
REPO_URL = "https://github.com/manojkumargudge/medical-imaging-ai.git"
REPO_REF = "main"
REPO = Path("/content/medical-imaging-ai")

if not REPO.exists():
    run("git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO)
commit = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print("repo commit", commit)
# On the path rather than installed: the repo pins are for local Python 3.11, and Colab's own
# packages (Pillow, PyYAML, pandas, torch with CUDA) are used here.
sys.path.insert(0, str(REPO))
os.chdir(REPO)

run(sys.executable, "-m", "pip", "install", "-q", "dvc==3.67.1")
remote = subprocess.run(
    ["dvc", "remote", "list"], cwd=REPO, capture_output=True, text=True, check=True
).stdout
print(remote)
assert "/content/drive/MyDrive/aiml02-dvc" in remote, "DVC remote is not the Drive folder"
# The cache hardlinks to the workspace instead of holding a second copy on the VM disk.
run("dvc", "config", "--local", "cache.type", "hardlink,copy", cwd=REPO)

run(sys.executable, "-m", "pip", "install", "-q", "monai==1.6.1", "nibabel==5.4.2")
import monai

print("torch", torch.__version__, "monai", monai.__version__)

## 3. Data to the VM disk, and the `seg_split` stage

`dvc repro seg_split` rebuilds the split from `dataset.json`. It must give the same file as the
committed `dvc.lock`; the cell stops if `dvc.lock` changes.

In [ ]:
from src.data import spleen
from src.data.preprocess import check_off_drive
from src.models.segmentation import load_seg_config

config = load_seg_config(REPO / "configs" / "segmentation.yaml")
check_off_drive(REPO / config.task_dir, "task_dir")  # train only from the VM disk
task_dir = spleen.download(REPO / "data" / "raw")  # md5 checked
print(task_dir, sorted(path.name for path in task_dir.iterdir()))

run("dvc", "repro", "seg_split", cwd=REPO)
changed = subprocess.run(["git", "diff", "--quiet", "dvc.lock"], cwd=REPO).returncode
if changed:
    run("git", "diff", "dvc.lock", cwd=REPO)
    raise RuntimeError("seg_split differs from the committed dvc.lock; stop and compare")
splits = spleen.read_split(REPO / config.split)  # prints counts, fails on overlap
cases = {name: len(rows) for name, rows in splits.items()}
print(cases)
assert all(cases.values())

## 4. Train (resumes after the latest checkpoint)

Sliding-window validation and a checkpoint every 10 epochs.

In [ ]:
from src.models import segmentation

print("checkpoints in", config.checkpoint_dir)
result = segmentation.train_segmentation(config)
print("best", {key: result["best"][key] for key in ("epoch", "val_mean_dice")})

## 5. Evaluate the best checkpoint on val and test

Dice per case. An existing report is not overwritten.

In [ ]:
import json
from functools import partial

from src.models.checkpoints import save_new, write_text

report = segmentation.evaluate_segmentation(config)
report["repo_commit"] = commit
out = config.checkpoint_dir / "reports" / "segmentation_metrics.json"
save_new(out, partial(write_text, text=json.dumps(report, indent=2) + "\n"))
for split, result in report["splits"].items():
    print(split, len(result["cases"]), "cases, mean Dice", result["mean_dice"])

## 6. Afterwards, on the laptop

Copy `My Drive/aiml02-runs/segmentation/reports/segmentation_metrics.json` to `reports/` and
commit it. The checkpoints stay on Drive.